In [1]:
import re
import numpy as np
from pathlib import Path
from typing import Tuple
import json

In [2]:
TARGET_NAME="LeNet-300-100"
TARGET_DIR="../results/1.LeNet-300-100"
TARGET_DATE="20260402"
TARGET_TYPE="canonical"
MODEL_ID1=1
MODEL_ID2=5
TARGET_CONFIG=f"freq_{MODEL_ID1}_{MODEL_ID2}.json"

In [3]:
def parse_c_header(file_path: str) -> Tuple[str, np.ndarray]:
    """
    Parse C header file and extract array name and data

    Args:
        file_path: Path to the C header file

    Returns:
        Tuple of (array_name, data_array)
    """
    with open(file_path, "r") as f:
        content = f.read()

    # Find array declaration
    array_pattern = r"unsigned\s+char\s+([a-zA-Z0-9_]+)\s*\[\]\s*=\s*\{"
    match = re.search(array_pattern, content)

    if not match:
        raise ValueError(f"Could not find array declaration in {file_path}")

    array_name = match.group(1)

    # Extract hex values
    hex_pattern = r"0x([0-9a-fA-F]{2})"
    hex_values = re.findall(hex_pattern, content)

    if not hex_values:
        raise ValueError(f"Could not find hex values in {file_path}")

    # Convert to numpy array
    data = np.array([int(h, 16) for h in hex_values], dtype=np.uint8)

    return array_name, data

In [4]:
CONFIG_PATH = Path(f"{TARGET_DIR}/{TARGET_DATE}/{TARGET_TYPE}/{TARGET_CONFIG}")

with open(CONFIG_PATH) as f:
    cfg = json.load(f)

best = cfg["best_encoding"]
HUFFMAN_CODES = best["huffman_codes"]  # symbol(str) → bitstring
MERGED_VALUES = set(best["merged_values"])
FALLBACK_SYMBOL = best["representative_value"] # special rare-delta token
FALLBACK_CODE = HUFFMAN_CODES[str(FALLBACK_SYMBOL)]
BITSLENGTH = best["bitsLen_symNum"]
MIN_BITS_LENGTH = best["min_code_length"]
MAX_BITS_LENGTH = best["max_code_length"]

# Build reverse map  bitstring → int symbol  (for decoding)
REVERSE_CODES = {v: int(k) for k, v in HUFFMAN_CODES.items()}

print(f"[config]  total symbols in Huffman table : {len(HUFFMAN_CODES)}")
print(f"[config]  merged (rare) values           : {len(MERGED_VALUES)}")
print(f"[config]  fallback code for 1000         : {FALLBACK_CODE}")
avg_bits = cfg["best_avg_bits"]
print(
    f"[config]  best avg bits/symbol           : {avg_bits:.4f}  "
    f"(vs 8 raw bits → {avg_bits/8*100:.1f}% of original)"
)

[config]  total symbols in Huffman table : 95
[config]  merged (rare) values           : 417
[config]  fallback code for 1000         : 00
[config]  best avg bits/symbol           : 7.5746  (vs 8 raw bits → 94.7% of original)


In [5]:
FALLBACK_SYMBOL

45

In [7]:
name1, data1 = parse_c_header(f"{TARGET_DIR}/models/{TARGET_NAME}_{MODEL_ID1}-quantized.h")
name2, data2 = parse_c_header(f"{TARGET_DIR}/models/{TARGET_NAME}_{MODEL_ID2}-quantized.h")

print(name1, "\n", name2)
print(f"Data 1 shape: {data1.shape}, Data 2 shape: {data2.shape}")

___models_1_LeNet_300_100_quantized_LeNet_300_100_1_quantized_tflite 
 ___models_1_LeNet_300_100_quantized_LeNet_300_100_5_quantized_tflite
Data 1 shape: (270720,), Data 2 shape: (270720,)


In [8]:
deltas = data2.astype(np.int16) - data1.astype(np.int16)
assert len(deltas) == len(
    data2.flat
), f"Length mismatch: delta={len(deltas)}, model2={len(data2.flat)}"

In [9]:
print(FALLBACK_SYMBOL)
print(FALLBACK_CODE)

45
00


In [10]:
import math
import zlib

# ── 1. Build raw bitstring
bits = []
huffman_hits = 0
fallback_hits = 0

for d, w2 in zip(deltas, data2.flat):
    key = str(int(d))
    if key in HUFFMAN_CODES and key != str(FALLBACK_SYMBOL):
        bits.append(HUFFMAN_CODES[key])
        huffman_hits += 1
    else:
        bits.append(FALLBACK_CODE)
        bits.append(f"{int(w2) & 0xFF:08b}")
        fallback_hits += 1

bitstring = "".join(bits)

# ── 2. Determine frame count and target size
# Each frame = 8 bytes. Last 4 bytes of the whole package = CRC-32.
# Payload capacity = (n * 8) - 4 bytes = (n * 64) - 32 bits.


def compute_n(bitstring: str) -> int:
    """Minimum number of 8-byte frames to hold the bitstring + 4-byte CRC."""
    raw_bytes = math.ceil(len(bitstring) / 8)  # bytes needed for bits alone
    total_bytes = raw_bytes + 4  # reserve 4 bytes for CRC
    return math.ceil(total_bytes / 8)  # round up to full frames


nb_frames = compute_n(bitstring)
target_bits = (nb_frames * 8 * 8) - 32  # total bits minus 32 CRC bits

# ── 3. Pad bitstring to exactly target_bits
pad = target_bits - len(bitstring)
bitstring += "0" * pad  # zero-pad to fill payload

# ── 4. Pack payload bits into bytes
payload = bytes(
    int(bitstring[i : i + 8], 2) for i in range(0, len(bitstring), 8)
)  # exactly (n*8 - 4) bytes

# ── 5. Append 4-byte CRC-32 placeholder, compute, then overwrite
package = bytearray(payload) + bytearray(4)  # full n*8 bytes, CRC = 0x00000000
# crc = zlib.crc32(bytes(package)) & 0xFFFFFFFF     # unsigned 32-bit, over entire package
# package[-4:] = crc.to_bytes(4, byteorder="big")   # overwrite placeholder with real CRC

package = bytes(package)  # final immutable bytes

# ── 6. Sanity checks
assert len(package) % 8 == 0, "Package must be a multiple of 8 bytes"
assert len(package) == nb_frames * 8, "Package size must equal n * 8 bytes"
assert len(package) >= len(payload) + 4, "CRC bytes must fit within package"

In [11]:
with open(f"{TARGET_DIR}/package_payload_{MODEL_ID1}_{MODEL_ID2}.bin", "wb") as f:
    f.write(package)